In [60]:
import pandas as pd
import sklearn as sk
import numpy as np
from catboost import CatBoostRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingRegressor

Firstly, we have to load the data

In [61]:
train_data = pd.read_csv('../data/train.csv')
test_data = pd.read_csv('../data/test.csv')

In [62]:
train_data.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [63]:
test_data.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate


The data seem pretty straightforward and they do not seem to need many modification, we could plot some graphs to get a better sense of the data as well as in the string values get the number of unique values. We do that because if we want to recreate that data as boolean columns, then it would not be beneficial if we had a huge number of columns. A lot of models do not accept or take way too long to train if the values in the columns are not numeric or boolean values.

In [64]:
unique_course_values = train_data['course'].unique()
unique_sleep_quality = train_data['sleep_quality'].unique()
unique_study_method = train_data['study_method'].unique()
unique_facility_rating = train_data['facility_rating'].unique()
unique_exam_difficulty = train_data['exam_difficulty'].unique()
print(f'Unique course values: {len(unique_course_values)}')
print(f'Unique sleep quality: {len(unique_sleep_quality)}')
print(f'Unique study method: {len(unique_study_method)}')
print(f'Unique facility rating: {len(unique_facility_rating)}')
print(f'Unique exam difficulty: {len(unique_facility_rating)}')

Unique course values: 7
Unique sleep quality: 3
Unique study method: 5
Unique facility rating: 3
Unique exam difficulty: 3


We can see that the number of unique values is not that high, so we could actually perform one hot encoding, something we should probably check is that we are not missing any unique values on the testing set that do not appear on the training one. Next we can view how many values are missing, so that we can take the correct approach to filling them, e.g with 0, median value, mean value etc

In [65]:
print(f'The na values for the training set:\n {train_data.isna().sum()}')
print(f'The na values for the testing set:\n {test_data.isna().sum()}')

The na values for the training set:
 id                  0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64
The na values for the testing set:
 id                  0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
dtype: int64


So there seem to be no missing values in the dataset, so we do not really have to fill any values, we will then go on to one hot encoding the feature columns.

In [66]:
feature_columns = train_data.select_dtypes(include=['object']).columns
train_data = pd.get_dummies(data=train_data, columns=feature_columns)
test_data = pd.get_dummies(data=test_data, columns=feature_columns)

In [67]:
train_data.head()

,id,age,study_hours,class_attendance,sleep_hours,exam_score,gender_female,gender_male,gender_other,course_b.com,...,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study,facility_rating_high,facility_rating_low,facility_rating_medium,exam_difficulty_easy,exam_difficulty_hard,exam_difficulty_moderate
0,0,21,7.91,98.8,4.9,78.3,True,False,False,False,...,False,False,True,False,False,True,False,True,False,False
1,1,18,4.95,94.8,4.7,46.7,False,False,True,False,...,False,False,False,True,False,False,True,False,False,True
2,2,20,4.68,92.6,5.8,99.0,True,False,False,False,...,False,False,False,False,True,False,False,False,False,True
3,3,19,2.00,49.5,8.3,63.9,False,True,False,False,...,True,False,False,False,True,False,False,False,False,True
4,4,23,7.65,86.9,9.6,100.0,False,True,False,False,...,False,False,False,True,True,False,False,True,False,False


In [68]:
test_data.head()

,id,age,study_hours,class_attendance,sleep_hours,gender_female,gender_male,gender_other,course_b.com,course_b.sc,...,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study,facility_rating_high,facility_rating_low,facility_rating_medium,exam_difficulty_easy,exam_difficulty_hard,exam_difficulty_moderate
0,630000,24,6.85,65.2,5.2,False,False,True,False,False,...,True,False,False,False,True,False,False,True,False,False
1,630001,18,6.61,45.0,9.3,False,True,False,False,False,...,False,False,False,False,False,True,False,True,False,False
2,630002,24,6.60,98.5,6.2,True,False,False,False,False,...,True,False,False,False,False,False,True,False,False,True
3,630003,24,3.03,66.3,5.7,False,True,False,False,False,...,False,True,False,False,False,False,True,False,False,True
4,630004,20,2.03,42.4,9.2,True,False,False,False,False,...,False,False,False,False,False,True,False,False,False,True


So now we can try to use different models and get our predictions.

In [69]:
y_train = train_data['exam_score']
X_train = train_data.drop('exam_score', axis=1)
student_ids = test_data['id']
X_test = test_data

X_test.drop('id', axis=1)
X_train.drop('id', axis=1)

,age,study_hours,class_attendance,sleep_hours,gender_female,gender_male,gender_other,course_b.com,course_b.sc,course_b.tech,...,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study,facility_rating_high,facility_rating_low,facility_rating_medium,exam_difficulty_easy,exam_difficulty_hard,exam_difficulty_moderate
0,21,7.91,98.8,4.9,True,False,False,False,True,False,...,False,False,True,False,False,True,False,True,False,False
1,18,4.95,94.8,4.7,False,False,True,False,False,False,...,False,False,False,True,False,False,True,False,False,True
2,20,4.68,92.6,5.8,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
3,19,2.00,49.5,8.3,False,True,False,False,True,False,...,True,False,False,False,True,False,False,False,False,True
4,23,7.65,86.9,9.6,False,True,False,False,False,False,...,False,False,False,True,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,18,4.86,70.7,4.1,True,False,False,False,False,True,...,False,True,False,False,True,False,False,False,False,True
629996,21,7.08,54.4,4.5,True,False,False,False,False,False,...,False,True,False,False,False,True,False,False,False,True
629997,24,0.64,44.2,4.3,False,True,False,False,False,False,...,False,False,True,False,False,True,False,False,False,True
629998,20,1.54,75.1,8.2,False,True,False,True,False,False,...,True,False,False,False,True,False,False,False,False,True


Firstly, we can try to use the cat boost regressor.

In [70]:
model = CatBoostRegressor()
model.fit(X_train, y_train)
y_predict = model.predict(X_test)

submission = pd.DataFrame({
    'id': student_ids,
    'exam_score': y_predict
})

submission.to_csv('../data/catboost_submission_final.csv', index=False)

Learning rate set to 0.113364
0:	learn: 17.5093259	total: 68.4ms	remaining: 1m 8s
1:	learn: 16.3056014	total: 120ms	remaining: 59.8s
2:	learn: 15.2867049	total: 176ms	remaining: 58.5s
3:	learn: 14.3852511	total: 228ms	remaining: 56.7s
4:	learn: 13.6043911	total: 291ms	remaining: 57.8s
5:	learn: 12.9252648	total: 347ms	remaining: 57.5s
6:	learn: 12.3495883	total: 407ms	remaining: 57.8s
7:	learn: 11.8647542	total: 457ms	remaining: 56.6s
8:	learn: 11.4544933	total: 506ms	remaining: 55.7s
9:	learn: 11.0973666	total: 563ms	remaining: 55.7s
10:	learn: 10.7933629	total: 612ms	remaining: 55s
11:	learn: 10.5301079	total: 658ms	remaining: 54.2s
12:	learn: 10.3077738	total: 709ms	remaining: 53.8s
13:	learn: 10.1175356	total: 761ms	remaining: 53.6s
14:	learn: 9.9499404	total: 815ms	remaining: 53.5s
15:	learn: 9.8064613	total: 872ms	remaining: 53.6s
16:	learn: 9.6847372	total: 930ms	remaining: 53.7s
17:	learn: 9.5773092	total: 988ms	remaining: 53.9s
18:	learn: 9.4865833	total: 1.04s	remaining: 53.6

Then, we can go on by trying out stacking, the cat boost is pretty good with these types of problems, but by using stacking and letting a lot of different models help predict the final output the results will probably improve

In [ ]:

base_models = {
    'xgboost': xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
    'lightgbm': lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1),
    'catboost': CatBoostRegressor(iterations=300, verbose=False, random_state=42),
    # 'rf': sk.ensemble.RandomForestRegressor(n_estimators=300, random_state=42),
    # 'extra_trees': sk.ensemble.ExtraTreesRegressor(n_estimators=200, random_state=42),
    'gradient_boost': GradientBoostingRegressor(n_estimators=200, random_state=42),
    'logistic': sk.linear_model.LogisticRegression(max_iter=5000, random_state=42),
    'naive_bayes': sk.naive_bayes.GaussianNB()
}

Kfold = sk.model_selection.KFold(n_splits=5, shuffle=True, random_state=42)

layer1_predictions = np.zeros((len(X_train), len(base_models)))

for fold_idx, (train_idx, test_idx) in enumerate(Kfold.split(X_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[test_idx]

    for model_idx, (name, model) in enumerate(base_models.items()):
        model.fit(X_fold_train, y_fold_train)
        if hasattr(model, 'predict_proba'):
            predictions = model.predict_proba(X_fold_val)[:, 1]
        else:
            predictions = model.predict(X_fold_val)

        layer1_predictions[test_idx, model_idx] = predictions

meta_model = sk.LogisticRegression(max_iter=5000, random_state=42)
meta_model.fit(layer1_predictions, y_train)

for name, model in base_models.items():
    model.fit(X_train, y_train)

layer1_test_predictions = np.zeros((len(X_train), len(base_models)))

for model_idx, (name, model) in enumerate(base_models.items()):
    if hasattr(model, 'predict_proba'):
        layer1_test_predictions[:, model_idx] = model.predict_proba(X_test)[:, 1]
    else:
        layer1_test_predictions[:, model_idx] = model.predict(X_test)

final_prediction = meta_model.predict(layer1_test_predictions)
print(final_prediction)

submission = pd.DataFrame({
    'id': student_ids,
    'exam_score': final_prediction
})
submission.to_csv('../data/StackedPredictions.csv', index=False)
print("\n✓ Submission saved!")
print(submission.head())